In [31]:
import pandas as pd
df = pd.read_csv('../data/permisos_construccion_2.csv', low_memory=False)

In [32]:
n = 15  

# 1. Diccionario con cantidad de valores distintos por columna
distinct_counts = {col: df[col].nunique(dropna=True) for col in df.columns}

# Convertir el diccionario a DataFrame y ordenarlo de menor a mayor
distinct_counts_df = (
    pd.DataFrame(list(distinct_counts.items()), columns=["Variable", "Distinct_Count"])
    .sort_values(by="Distinct_Count", ascending=True)
    .reset_index(drop=True)
)


# 2. Diccionario con detalle de valores para columnas con menos de n valores distintos
small_categories = {
    col: df[col].value_counts(dropna=True).to_dict()
    for col, count in distinct_counts.items()
    if count < n
}


In [33]:
distinct_counts_df

,Variable,Distinct_Count
0,Structural Notification,1
1,Voluntary Soft-Story Retrofit,1
2,Fire Only Permit,1
3,Site Permit,1
4,TIDF Compliance,2
5,Proposed Construction Type Description,5
6,Proposed Construction Type,5
7,Existing Construction Type,7
8,Plansets,8
9,Existing Construction Type Description,8


In [34]:
small_categories

{'Permit Type': {8: 178852,
  3: 14664,
  4: 2892,
  2: 950,
  6: 600,
  7: 511,
  1: 350,
  5: 91},
 'Permit Type Definition': {'otc alterations permit': 178836,
  'additions alterations or repairs': 14663,
  'sign - erect': 2892,
  'new construction wood frame': 950,
  'demolitions': 600,
  'wall or painted sign': 511,
  'new construction': 347,
  'grade or quarry or fill or excavate': 91,
  ' otc alterations permit ': 8,
  'otc alterations permit #': 8,
  'new construction #': 2,
  ' new construction ': 1,
  ' additions alterations or repairs ': 1},
 'Current Status': {'complete': 97081,
  'issued': 83563,
  'filed': 12045,
  'withdrawn': 1754,
  'cancelled': 1536,
  'expired': 1370,
  'approved': 733,
  'reinstated': 563,
  'suspend': 193,
  'revoked': 50,
  'plancheck': 16,
  'incomplete': 2,
  'disapproved': 2,
  'appeal': 2},
 'Structural Notification': {'Y': 6922},
 'Voluntary Soft-Story Retrofit': {'Y': 35},
 'Fire Only Permit': {'Y': 18828},
 'Plansets': {2.0: 98093,
  0.0: 6

In [35]:
small_df = (
    pd.DataFrame(
        [(var, valor, cuenta) 
         for var, subdict in small_categories.items() 
         for valor, cuenta in subdict.items()],
        columns=["Variable", "Value", "Count"]
    )
    .sort_values(by=["Variable", "Count"], ascending=[True, False])
    .reset_index(drop=True)
)

In [36]:
small_df

,Variable,Value,Count
0,Current Status,complete,97081
1,Current Status,issued,83563
2,Current Status,filed,12045
3,Current Status,withdrawn,1754
4,Current Status,cancelled,1536
...,...,...,...
83,Supervisor District,veinte,1
84,Supervisor District,diez,1
85,TIDF Compliance,P,1
86,TIDF Compliance,Y,1


In [37]:
df[["Proposed Construction Type Description", "Proposed Construction Type", "Existing Construction Type Description", "Existing Construction Type"]].value_counts(dropna=False).reset_index(name='count')

,Proposed Construction Type Description,Proposed Construction Type,Existing Construction Type Description,Existing Construction Type,count
0,wood frame (5),5.0,wood frame (5),5.0,111938
1,NaN,NaN,NaN,NaN,39625
2,constr type 1,1.0,constr type 1,1.0,26815
3,constr type 3,3.0,constr type 3,3.0,9085
4,constr type 2,2.0,constr type 2,2.0,3672
5,wood frame (5),5.0,NaN,NaN,2412
6,NaN,NaN,wood frame (5),5.0,1344
7,NaN,NaN,constr type 1,1.0,1239
8,constr type 1,1.0,NaN,NaN,998
9,NaN,NaN,constr type 3,3.0,550


In [38]:
df[["Proposed Construction Type", "Existing Construction Type"]].value_counts(dropna=False).reset_index(name='count')


,Proposed Construction Type,Existing Construction Type,count
0,5.0,5.0,111942
1,NaN,NaN,39625
2,1.0,1.0,26816
3,3.0,3.0,9086
4,2.0,2.0,3672
5,5.0,NaN,2412
6,NaN,5.0,1344
7,NaN,1.0,1239
8,1.0,NaN,998
9,NaN,3.0,550


# Análisis de columnas escogidas

Se tomara la decision de eliminar las columnas "Proposed Construction Type Description" y "Existing Construction Type Description" ya que son redundantes con las columnas "Proposed Construction Type" y "Existing Construction Type" respectivamente. Además se deberá normalizar y corregir alguna de las columnas los valores. Para esto se ejecutara el siguiente código:

### Función del código

El código realiza un proceso de **normalización y consistencia** entre las columnas *Existing Construction Type* y *Proposed Construction Type* en el dataset:

1. **Limpieza de Existing Construction Type**  
   - Convierte los valores a formato numérico.  
   - Mantiene únicamente códigos válidos (1.0, 2.0, 3.0, 4.0, 5.0).  
   - Si hay descripciones de tipo `"wood frame (5)"`, las corrige a 5.0.  
   - Todo valor que no sea reconocible se transforma en `NaN`.

2. **Consistencia con Proposed Construction Type**  
   - Convierte también la columna a formato numérico.  
   - Si una fila tiene un valor válido en *Existing Construction Type*, se asegura de que *Proposed Construction Type* tenga el mismo valor.  
   - Si *Existing Construction Type* está vacío (`NaN`), *Proposed* no se modifica y puede permanecer vacío.

En resumen, el código garantiza que los tipos de construcción se mantengan **limpios, coherentes y sincronizados**, utilizando la información de *Existing* como referencia principal para completar *Proposed*.

In [39]:
import numpy as np

# --- Normalizar Existing Construction Type ---
# Convertir a numérico, valores inválidos a NaN
df["Existing Construction Type"] = pd.to_numeric(
    df["Existing Construction Type"], errors="coerce"
)

valid_existing = {1.0, 2.0, 3.0, 4.0, 5.0}
mask_invalid_exist = ~df["Existing Construction Type"].isin(valid_existing)

# Normalizar la Description para comparar de forma robusta
norm_desc = (
    df["Existing Construction Type Description"]
      .astype("string")
      .str.strip()
      .str.lower()
)

# Coincidencia con "wood frame (5)"
mask_wood5 = mask_invalid_exist & norm_desc.str.contains(
    r"^wood\s*frame\s*\(\s*5\s*\)$", regex=True, na=False
)

# Asignar 5.0 cuando la descripción indique wood frame (5)
df.loc[mask_wood5, "Existing Construction Type"] = 5.0

# El resto de los inválidos -> NaN
df.loc[mask_invalid_exist & ~mask_wood5, "Existing Construction Type"] = np.nan


# --- Nueva lógica: Proposed copia Existing cuando Existing no es NaN ---
df["Proposed Construction Type"] = pd.to_numeric(
    df["Proposed Construction Type"], errors="coerce"
)

mask_copy = df["Existing Construction Type"].notna()
df.loc[mask_copy, "Proposed Construction Type"] = df.loc[mask_copy, "Existing Construction Type"]

In [40]:
df[["Proposed Construction Type", "Existing Construction Type"]].value_counts(dropna=False).reset_index(name='count')

,Proposed Construction Type,Existing Construction Type,count
0,5.0,5.0,113354
1,NaN,NaN,39626
2,1.0,1.0,28073
3,3.0,3.0,9664
4,2.0,2.0,4068
5,5.0,NaN,2412
6,1.0,NaN,998
7,4.0,4.0,381
8,3.0,NaN,235
9,2.0,NaN,93


In [41]:
def merge_proposed_existing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Asegurar que sean numéricos
    df["Proposed Construction Type"] = pd.to_numeric(df["Proposed Construction Type"], errors="coerce")
    df["Existing Construction Type"] = pd.to_numeric(df["Existing Construction Type"], errors="coerce")

    # Crear nueva columna combinada
    df["Construction Type"] = np.where(
        df["Existing Construction Type"].notna(),
        "E " + df["Existing Construction Type"].astype(str),
        np.where(
            df["Proposed Construction Type"].notna(),
            "P " + df["Proposed Construction Type"].astype(str),
            np.nan
        )
    )

    return df

In [42]:
df = merge_proposed_existing(df)

In [43]:
df["Construction Type"].value_counts(dropna=False).reset_index(name='count')

,Construction Type,count
0,E 5.0,113354
1,NaN,39626
2,E 1.0,28073
3,E 3.0,9664
4,E 2.0,4068
5,P 5.0,2412
6,P 1.0,998
7,E 4.0,381
8,P 3.0,235
9,P 2.0,93


In [44]:
df = df.drop(columns=[
    "Proposed Construction Type Description",
    "Existing Construction Type Description",
    "Proposed Construction Type",
    "Existing Construction Type"
], errors="ignore")




### Decisión final sobre las columnas de construcción

Se decidió **unificar y simplificar** la información proveniente de las cuatro columnas iniciales:

- **`Proposed Construction Type Description`**  
- **`Proposed Construction Type`**  
- **`Existing Construction Type Description`**  
- **`Existing Construction Type`**

Tras un proceso de limpieza y normalización, se eliminaron las columnas redundantes de *Description* y se consolidaron las columnas de *Proposed* y *Existing* en una **única columna final** llamada `Construction Type`.  

En esta columna:

- Si existía un valor válido en **Existing**, se conserva con el prefijo **`E`** (ej: `E 3.0`).  
- Si *Existing* estaba vacío pero *Proposed* tenía valor, se conserva con el prefijo **`P`** (ej: `P 1.0`).  
- Si ambas estaban vacías, se deja como `NaN`.  

De esta manera, la información queda unificada, coherente y lista para su uso en el análisis, evitando duplicaciones y garantizando consistencia.

In [45]:
df.head(2)

,Permit Number,Permit Type,Permit Type Definition,Permit Creation Date,Block,Lot,Street Number,Street Number Suffix,Street Name,Street Suffix,...,Proposed Units,Plansets,TIDF Compliance,Site Permit,Supervisor District,Neighborhoods - Analysis Boundaries,Zipcode,Location,Record ID,Construction Type
0,M788927,8,otc alterations permit,05/23/2017,0215,001,1333,NaN,jOnEs,St,...,NaN,NaN,NaN,NaN,3.0,Nob Hill,94109.0,"(37.79362102799777, -122.41488237355445)",1464153232862,NaN
1,201305318356,8,otc alterations permit,05/31/2013,1810,017A,1483,NaN,43rD,Av,...,1.0,2.0,NaN,NaN,4.0,Sunset/Parkside,94122.0,"(37.759041020475465, -122.50286985467523)",1306559115258,E 5.0


# Permit Type Definition

Corrección de algunos valores que ponía como distintos cuando eran iguales

In [46]:
# 1. Limpiar texto: a string, minúsculas, quitar espacios extra
df["Permit Type Definition"] = (
    df["Permit Type Definition"]
      .astype("string")
      .str.lower()
      .str.strip()                      # quita espacios adelante y atrás
      .str.replace(r"\s+", " ", regex=True)  # reduce múltiples espacios a uno
)

# 2. Mapear variantes conocidas a la etiqueta estándar
map_dict = {
    "otc alterations permit #": "otc alterations permit",
    "otc alterations permit": "otc alterations permit",
    "additions alterations or repairs": "additions alterations or repairs",
    "new construction wood frame": "new construction wood frame",
    "new construction": "new construction",
    "new construction #": "new construction",
    "sign - erect": "sign - erect",
    "demolitions": "demolitions",
    "wall or painted sign": "wall or painted sign",
    "grade or quarry or fill or excavate": "grade or quarry or fill or excavate"
}

df["Permit Type Definition"] = df["Permit Type Definition"].replace(map_dict)

# Supervisor District
Corrección de formato

In [47]:
# Normalizar primero a string para poder mapear palabras
df["Supervisor District"] = (
    df["Supervisor District"]
      .astype("string")
      .str.strip()
      .str.lower()
)

# Reemplazar palabras por números en string
map_dict = {
    "quince": "15.0",
    "veinte": "20.0",
    "diez": "10.0"
}
df["Supervisor District"] = df["Supervisor District"].replace(map_dict)

# Convertir a float nativo de Python (que pandas representará como 1.0, 15.0, etc.)
df["Supervisor District"] = df["Supervisor District"].apply(lambda x: float(x) if pd.notna(x) else x)

# Hasta n < 15 estan todos bien ya

In [48]:
n = 50  

# 1. Diccionario con cantidad de valores distintos por columna
distinct_counts = {col: df[col].nunique(dropna=True) for col in df.columns}

# Convertir el diccionario a DataFrame y ordenarlo de menor a mayor
distinct_counts_df = (
    pd.DataFrame(list(distinct_counts.items()), columns=["Variable", "Distinct_Count"])
    .sort_values(by="Distinct_Count", ascending=True)
    .reset_index(drop=True)
)


# 2. Diccionario con detalle de valores para columnas con menos de n valores distintos
small_categories = {
    col: df[col].value_counts(dropna=True).to_dict()
    for col, count in distinct_counts.items()
    if  15 < count < n
}

In [49]:
small_categories

{'Street Number Suffix': {'A': 1501,
  'B': 291,
  'V': 228,
  'C': 56,
  'E': 28,
  'F': 24,
  'G': 12,
  'D': 11,
  'K': 11,
  'H': 11,
  'R': 10,
  'L': 10,
  'J': 9,
  'I': 7,
  'P': 3,
  'N': 2,
  '½': 1,
  '0': 1},
 'Street Suffix': {'St': 138365,
  'Av': 43222,
  'Bl': 3555,
  'Wy': 3540,
  'Dr': 3267,
  'Tr': 1466,
  'Ct': 667,
  'Pl': 538,
  'Rd': 389,
  'Ln': 354,
  'Hy': 240,
  'Pz': 210,
  'Pk': 128,
  'Cr': 97,
  'Al': 83,
  'Wk': 9,
  'Rw': 5,
  'No': 2,
  'So': 2,
  'Sw': 2,
  'Hl': 1},
 'Neighborhoods - Analysis Boundaries': {'Financial District/South Beach': 21816,
  'Mission': 14682,
  'Sunset/Parkside': 10207,
  'West of Twin Peaks': 8740,
  'Castro/Upper Market': 8527,
  'Pacific Heights': 8508,
  'Marina': 8244,
  'Outer Richmond': 7855,
  'Noe Valley': 7844,
  'South of Market': 7573,
  'Bernal Heights': 6068,
  'Nob Hill': 6010,
  'Haight Ashbury': 5799,
  'Inner Sunset': 5776,
  'Bayview Hunters Point': 5670,
  'Russian Hill': 5495,
  'Hayes Valley': 5489,
  'Te

# Streets
Todos los parámetros de dirección en un solo campo "Address"

In [50]:
import re
import pandas as pd

# Asegurar existencia de columnas (si falta alguna, la creamos vacía)
cols_src = [
    "Street Name", "Street Number", "Street Number Suffix", "Street Suffix",
    "Lot", "Block", "Zipcode",
    "Unit", "Unit Suffix", "Neighborhoods - Analysis Boundaries"
]
for c in cols_src:
    if c not in df.columns:
        df[c] = pd.NA

# --- Normalizaciones ---
# Mayúsculas y limpieza de texto
def _to_upper_str(s):
    return s.astype("string").str.strip().str.upper()

df["Street Name"] = _to_upper_str(df["Street Name"])
df["Street Suffix"] = _to_upper_str(df["Street Suffix"])
df["Street Number Suffix"] = _to_upper_str(df["Street Number Suffix"])
df["Unit"] = _to_upper_str(df["Unit"])
df["Unit Suffix"] = _to_upper_str(df["Unit Suffix"])
df["Neighborhoods - Analysis Boundaries"] = _to_upper_str(df["Neighborhoods - Analysis Boundaries"])

# Quitar ".0" heredado de floats y dejar string limpio
def _norm_token(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s if s != "" else pd.NA

for col in ["Street Number", "Lot", "Block", "Unit", "Unit Suffix"]:
    df[col] = df[col].apply(_norm_token).astype("string")

# Zipcode: conservar como string; quitar no-dígitos y formatear 5 ó 9 (XXXXX-XXXX)
def _norm_zip(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    digits = re.sub(r"\D", "", s)
    if digits == "":
        return pd.NA
    if len(digits) >= 9:
        return f"{digits[:5]}-{digits[5:9]}"
    elif len(digits) >= 5:
        return digits[:5]
    else:
        return digits  # si es más corto, lo dejamos tal cual

df["Zipcode"] = df["Zipcode"].apply(_norm_zip).astype("string")

# --- Construcción de Address ---
# 1) Calle: "NUM NUM_SFX STREET_NAME STREET_SFX"
def _compose_street(r):
    parts = [r["Street Number"], r["Street Number Suffix"], r["Street Name"], r["Street Suffix"]]
    parts = [p for p in parts if pd.notna(p) and p != ""]
    return " ".join(parts) if parts else pd.NA

street_part = (
    df.apply(_compose_street, axis=1)
      .astype("string")
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

# 2) Unidad: "UNIT {Unit}{UnitSuffix}" (si existe)
def _compose_unit(u, usfx):
    parts = [p for p in [u, usfx] if pd.notna(p) and p != ""]
    if parts:
        token = "".join(parts)  # 2 + B -> 2B | 2 + ½ -> 2½
        return f"UNIT {token}"
    return pd.NA

unit_full = [
    _compose_unit(u, us)
    for u, us in zip(df["Unit"], df["Unit Suffix"])
]
unit_full = pd.Series(unit_full, index=df.index, dtype="string")

# 3) Street + Unit (sin coma entre ellos: "123 MAIN ST UNIT 2B")
street_plus_unit = street_part.where(unit_full.isna() | (unit_full == ""), street_part + " " + unit_full)

# 4) Partes auxiliares con etiquetas claras
block_part = df["Block"].apply(lambda v: f"BLOCK {v}" if pd.notna(v) and v != "" else pd.NA).astype("string")
lot_part   = df["Lot"].apply(lambda v: f"LOT {v}"   if pd.notna(v) and v != "" else pd.NA).astype("string")
neigh_part = df["Neighborhoods - Analysis Boundaries"].astype("string").apply(
    lambda v: v if pd.notna(v) and v != "" else pd.NA
)
zip_part   = df["Zipcode"].apply(lambda v: v if pd.notna(v) and v != "" else pd.NA).astype("string")

# 5) Unir con comas (omitiendo vacíos)
def _join_with_commas(*vals):
    vals = [v for v in vals if pd.notna(v) and str(v) != ""]
    return ", ".join(vals) if vals else pd.NA

df["Address"] = [
    _join_with_commas(su, b, l, n, z)
    for su, b, l, n, z in zip(street_plus_unit, block_part, lot_part, neigh_part, zip_part)
]

# Limpieza final de espacios/comas accidentales
df["Address"] = (
    df["Address"]
      .astype("string")
      .str.replace(r"\s+", " ", regex=True)
      .str.replace(r"\s+,", ",", regex=True)
      .str.replace(r",\s*,", ", ", regex=True)
      .str.strip()
)

df = df.drop(columns=cols_src)

In [51]:
df["Address"].value_counts(dropna=False).reset_index(name='count')

,Address,count
0,"101 CALIFORNIA ST, BLOCK 0263, LOT 011, FINANC...",330
1,"1455 MARKET ST, BLOCK 3507, LOT 040, SOUTH OF ...",276
2,"3251 20TH AV, BLOCK 7295, LOT 021, LAKESHORE, ...",245
3,"1 MARKET ST, BLOCK 3713, LOT 007, FINANCIAL DI...",225
4,"1355 MARKET ST, BLOCK 3508, LOT 001, SOUTH OF ...",220
...,...,...
81876,"2026 33RD AV, BLOCK 2152, LOT 035, SUNSET/PARK...",1
81877,"76 GATES ST, BLOCK 5625, LOT 015, BERNAL HEIGH...",1
81878,"222 HERMANN ST, BLOCK 0868, LOT 008A, HAYES VA...",1
81879,"75 HARTFORD ST UNIT 0, BLOCK 3582, LOT 025, CA...",1


# Existing Use y Proposed Use

In [52]:
# Contar combinaciones únicas de Existing Use y Proposed Use
combos = (
    df[["Existing Use", "Proposed Use"]]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

# Número de combinaciones distintas
num_unique = combos.shape[0]
combos

,Existing Use,Proposed Use,count
0,1 family dwelling,1 family dwelling,45381
1,apartments,apartments,40447
2,NaN,NaN,38806
3,office,office,23403
4,2 family dwelling,2 family dwelling,20133
...,...,...,...
641,vacant lot,animal sale or care,1
642,lending institution,clinics-medic/dental,1
643,lending institution,church,1
644,lending institution,apartments,1


# Existing Units y Proposed Units

In [53]:
# Contar combinaciones únicas de Existing Units y Proposed Units
combos_units = (
    df[["Existing Units", "Proposed Units"]]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

# Número de combinaciones distintas
num_unique_units = combos_units.shape[0]

combos_units

,Existing Units,Proposed Units,count
0,NaN,NaN,48527
1,1.0,1.0,45940
2,0.0,0.0,26601
3,2.0,2.0,20910
4,3.0,3.0,8380
...,...,...,...
1072,84.0,NaN,1
1073,82.0,83.0,1
1074,82.0,81.0,1
1075,80.0,78.0,1


In [54]:
mask_valid = df["Existing Units"].notna() & df["Proposed Units"].notna()
mask_diff = mask_valid & (df["Existing Units"] != df["Proposed Units"])

num_diff = mask_diff.sum()
total_valid = mask_valid.sum()

print(f"Difieren en {num_diff} de {total_valid} registros válidos ({num_diff/total_valid:.2%})")

# Opcional: ver las combinaciones distintas y sus frecuencias
diff_combos = (
    df.loc[mask_diff, ["Existing Units", "Proposed Units"]]
      .value_counts()
      .reset_index(name="count")
)

diff_combos

Difieren en 4346 de 144979 registros válidos (3.00%)


,Existing Units,Proposed Units,count
0,1.0,2.0,1127
1,2.0,3.0,710
2,0.0,1.0,197
3,6.0,7.0,155
4,0.0,2.0,153
...,...,...,...
401,32.0,33.0,1
402,32.0,34.0,1
403,32.0,38.0,1
404,32.0,42.0,1


# Record ID 
Dice que no sirve para este analisis

In [55]:
df = df.drop(columns="Record ID")

In [56]:
# Seleccionar solo las columnas de interés
subset = df[["Site Permit", "Fire Only Permit", "Permit Type"]]

# Contar combinaciones únicas
combos = (
    subset.value_counts(dropna=False)
          .reset_index(name="count")
)

# Número de tuplas distintas
num_unique = combos.shape[0]

print(f"Número de tuplas distintas: {num_unique}")
display(combos)

Número de tuplas distintas: 15


,Site Permit,Fire Only Permit,Permit Type,count
0,NaN,NaN,8,161263
1,NaN,Y,8,17588
2,NaN,NaN,3,9106
3,Y,NaN,3,4319
4,NaN,NaN,4,2892
5,NaN,Y,3,1239
6,Y,NaN,2,718
7,NaN,NaN,6,600
8,NaN,NaN,7,510
9,Y,NaN,1,322


Special Permit
Se unifico las variables Site Permit y Fire Only Permit en una sola columna Special Permit

In [57]:
def _permit_combo(row):
    site = row["Site Permit"] == "Y"
    fire = row["Fire Only Permit"] == "Y"
    if site and fire:
        return "SITE & FIRE"
    elif site:
        return "SITE PERMIT"
    elif fire:
        return "FIRE ONLY PERMIT"
    else:
        return pd.NA

# Crear la columna unificada
df["Special Permit"] = df.apply(_permit_combo, axis=1)

# Eliminar las columnas originales
df = df.drop(columns=["Site Permit", "Fire Only Permit"])

In [58]:
df

,Permit Number,Permit Type,Permit Type Definition,Permit Creation Date,Description,Current Status,Current Status Date,Filed Date,Issued Date,Completed Date,...,Existing Units,Proposed Use,Proposed Units,Plansets,TIDF Compliance,Supervisor District,Location,Construction Type,Address,Special Permit
0,M788927,8,otc alterations permit,05/23/2017,street space,issued,05/23/2017,05/23/2017,05/23/2017,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,"(37.79362102799777, -122.41488237355445)",NaN,"1333 JONES ST, BLOCK 0215, LOT 001, NOB HILL, ...",<NA>
1,201305318356,8,otc alterations permit,05/31/2013,"remodel kitchen: replace countertop, cabinets,...",complete,08/28/2013,05/31/2013,06/03/2013,08/28/2013,...,1.0,1 family dwelling,1.0,2.0,NaN,4.0,"(37.759041020475465, -122.50286985467523)",E 5.0,"1483 43RD AV, BLOCK 1810, LOT 017A, SUNSET/PAR...",<NA>
2,201705106205,8,otc alterations permit,05/10/2017,replacement of 4 windows; 2 located in lightwe...,issued,05/11/2017,05/10/2017,05/11/2017,NaN,...,1.0,1 family dwelling,1.0,0.0,NaN,9.0,"(37.73778863007536, -122.41197863877355)",E 5.0,"431 PRENTISS ST, BLOCK 5700, LOT 027, BERNAL H...",<NA>
3,201410279983,8,otc alterations permit,10/27/2014,two kitchens & two bathrooms remodel. replace ...,complete,12/31/2014,10/27/2014,10/27/2014,12/31/2014,...,2.0,2 family dwelling,2.0,0.0,NaN,5.0,"(37.78762264983362, -122.43099126735969)",E 5.0,"2020 BUSH ST, BLOCK 0661, LOT 005, PACIFIC HEI...",<NA>
4,201310280388,8,otc alterations permit,10/28/2013,reroofing,issued,10/28/2013,10/28/2013,10/28/2013,NaN,...,4.0,apartments,4.0,0.0,NaN,9.0,"(37.75275550565926, -122.41707462095194)",E 5.0,"871 CAPP ST, BLOCK 3642, LOT 051A, MISSION, 94110",<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198905,201604053958,3,additions alterations or repairs,04/05/2016,"ti for space 129,130, 132 partitions, mep, she...",complete,11/22/2016,04/05/2016,07/18/2016,11/22/2016,...,0.0,retail sales,0.0,2.0,NaN,7.0,"(37.728556952954136, -122.47676641508518)",E 2.0,"3251 20TH AV, BLOCK 7295, LOT 021, LAKESHORE, ...",<NA>
198906,201510270880,8,otc alterations permit,10/27/2015,"install new 2"" underground combination fire ma...",issued,10/27/2015,10/27/2015,10/27/2015,NaN,...,1.0,1 family dwelling,1.0,2.0,NaN,10.0,"(37.76328445631136, -122.40287014554292)",E 5.0,"1919 MARIPOSA ST, BLOCK 4009, LOT 001A, POTRER...",FIRE ONLY PERMIT
198907,201607293741,8,otc alterations permit,07/29/2016,replace rotten wooden moldings in kind front s...,issued,07/29/2016,07/29/2016,07/29/2016,NaN,...,1.0,1 family dwelling,1.0,0.0,NaN,10.0,"(37.75260530951628, -122.40400191084352)",E 5.0,"1341 SAN BRUNO AV, BLOCK 4262, LOT 020, MISSIO...",<NA>
198908,201701066691,8,otc alterations permit,01/06/2017,revision to pa 2016-0926-8773; remove (e) stor...,issued,02/01/2017,01/06/2017,02/01/2017,NaN,...,9.0,retail sales,9.0,2.0,NaN,3.0,"(37.79800446861674, -122.4080339831039)",E 5.0,"660 BROADWAY, BLOCK 0146, LOT 007, CHINATOWN, ...",<NA>


## Existing Units y Proposed Units

In [1]:
# Existing Units: NaN → 0
if "Existing Units" in df.columns:
    df["Existing Units"] = df["Existing Units"].fillna(0).astype(int)

# Proposed Units: 0 → NaN
if "Proposed Units" in df.columns:
    df["Proposed Units"] = df["Proposed Units"].replace(0, np.nan)


NameError: name 'df' is not defined

## Estimated Cost and Revised Cost

In [ ]:
for col in ["Estimated Cost", "Revised Cost"]:
    if col in df.columns:
        df[col] = df[col].replace([0, 1], np.nan)

## Plansets

In [ ]:
if "Plansets" in df.columns:
    df["Plansets"] = df["Plansets"].fillna(0).astype(int)

## Number of Existing Stories and Number of Proposed Stories

In [ ]:
# Existing Stories: NaN -> 0
if "Number of Existing Stories" in df.columns:
    df["Number of Existing Stories"] = (
        df["Number of Existing Stories"].fillna(0).astype(int)
    )

# Proposed Stories: 0 -> NaN
if "Number of Proposed Stories" in df.columns:
    df["Number of Proposed Stories"] = (
        df["Number of Proposed Stories"].replace(0, np.nan).astype("Int64")
    )